Evaluation Part II

Evaluate LLM responses where there isn't a single "right answer."

Setup

Load the API key and relevant Python libaries.

In this course, we've provided some code that loads the OpenAI API key for you.

In [1]:
!pip install groq
from google.colab import userdata
from groq import Groq

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.4 MB/s eta 0:00:00


In [2]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def get_completion_from_messages(
    messages,
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


Run through the end-to-end system to answer the user query
These helper functions are running the chain of promopts that you saw in the earlier videos.

In [13]:
class Utils:
    def get_products_and_category(self):
        # Placeholder for actual product data
        return {
            "smartx pro phone": {"category": "Smartphones", "description": "A high-performance smartphone.", "price": "$999", "features": "6.1-inch display, 128GB storage, 12MP camera"},
            "fotosnap camera": {"category": "Cameras", "description": "A professional DSLR camera.", "price": "$1200", "features": "24MP sensor, 4K video, interchangeable lenses"},
            "tv": {"category": "Televisions", "description": "Various models of smart TVs.", "price": "$700-​$2000", "features": "Smart TV features, 4K resolution, HDR"}
        }

    def find_category_and_product_only(self, user_input, products_data):
        found_products = []
        user_input_lower = user_input.lower()
        for product_name, product_info in products_data.items():
            if product_name in user_input_lower:
                found_products.append(f"Category: {product_info['category']}, Product: {product_name}")
            elif product_info['category'].lower() in user_input_lower:
                found_products.append(f"Category: {product_info['category']}, Product: {product_name}")
        # Return a string that can be parsed by read_string_to_list
        return "; ".join(found_products)

    def read_string_to_list(self, input_string):
        if not input_string:
            return []
        return [item.strip() for item in input_string.split(';') if item.strip()]

    def generate_output_string(self, product_list):
        output_lines = []
        for item in product_list:
            parts = item.split(', ')
            category = parts[0].split(': ')[1]
            product = parts[1].split(': ')[1]
            products_data = self.get_products_and_category()

            # Retrieve full product details including price and features
            full_product_info = products_data.get(product, {})
            description = full_product_info.get('description', 'No description available.')
            price = full_product_info.get('price', 'N/A')
            features = full_product_info.get('features', 'N/A')

            output_lines.append(f"Product: {product}, Category: {category}, Description: {description}, Price: {price}, Features: {features}")
        return "\n".join(output_lines)

    def get_mentioned_product_info(self, category_and_product_list):
        # Reusing generate_output_string to format product information
        return self.generate_output_string(category_and_product_list)

    def answer_user_msg(self, user_msg, product_info):
        messages = [
            {"role": "system", "content": "You are a helpful assistant. You have access to product information. Use this information to answer user queries concisely and accurately. If a product is mentioned in the user query, provide its details. If a category is mentioned, list products from that category. If a product is not found, state that you don't have information on it."},
            {"role": "user", "content": f"User query: {user_msg}\nProduct information: {product_info}"}
        ]
        return get_completion_from_messages(messages)

utils = Utils()

In [14]:
customer_msg = f"""
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs or TV related products do you have?"""

all_products_data = utils.get_products_and_category()
products_by_category = utils.find_category_and_product_only(customer_msg, all_products_data)
category_and_product_list = utils.read_string_to_list(products_by_category)

product_info = utils.get_mentioned_product_info(category_and_product_list)
assistant_answer = utils.answer_user_msg(user_msg=customer_msg,
                                                   product_info=product_info)

print(assistant_answer)

The SmartX Pro Phone is a high-performance smartphone with a 6.1-inch display, 128GB storage, and a 12MP camera, priced at $999.

The Fotosnap Camera is a professional DSLR camera with a 24MP sensor, 4K video capability, and interchangeable lenses, priced at $1200.

As for TVs or TV-related products, we have a range of smart TVs with 4K resolution, HDR, and smart TV features, priced between $700 and $2000.


In [15]:
print(assistant_answer)

The SmartX Pro Phone is a high-performance smartphone with a 6.1-inch display, 128GB storage, and a 12MP camera, priced at $999.

The Fotosnap Camera is a professional DSLR camera with a 24MP sensor, 4K video capability, and interchangeable lenses, priced at $1200.

As for TVs or TV-related products, we have a range of smart TVs with 4K resolution, HDR, and smart TV features, priced between $700 and $2000.


Evaluate the LLM's answer to the user with a rubric, based on the extracted product information


In [16]:
cust_prod_info = {
    'customer_msg': customer_msg,
    'context': product_info
}

In [17]:
def eval_with_rubric(test_set, assistant_answer):

    cust_msg = test_set['customer_msg']
    context = test_set['context']
    completion = assistant_answer

    system_message = """\
    You are an assistant that evaluates how well the customer service agent \
    answers a user question by looking at the context that the customer service \
    agent is using to generate its response.
    """

    user_message = f"""\
You are evaluating a submitted answer to a question based on the context \
that the agent uses to answer the question.
Here is the data:
    [BEGIN DATA]
    ************
    [Question]: {cust_msg}
    ************
    [Context]: {context}
    ************
    [Submission]: {completion}
    ************
    [END DATA]

Compare the factual content of the submitted answer with the context. \
Ignore any differences in style, grammar, or punctuation.
Answer the following questions:
    - Is the Assistant response based only on the context provided? (Y or N)
    - Does the answer include information that is not provided in the context? (Y or N)
    - Is there any disagreement between the response and the context? (Y or N)
    - Count how many questions the user asked. (output a number)
    - For each question that the user asked, is there a corresponding answer to it?
      Question 1: (Y or N)
      Question 2: (Y or N)
      ...
      Question N: (Y or N)
    - Of the number of questions asked, how many of these questions were addressed by the answer? (output a number)
"""

    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_message}
    ]

    response = get_completion_from_messages(messages)
    return response

In [18]:
evaluation_output = eval_with_rubric(cust_prod_info, assistant_answer)
print(evaluation_output)

To evaluate the submitted answer based on the provided context:

1. **Is the Assistant response based only on the context provided?** (Y or N)
   - Y

2. **Does the answer include information that is not provided in the context?** (Y or N)
   - N

3. **Is there any disagreement between the response and the context?** (Y or N)
   - N

4. **Count how many questions the user asked.** (output a number)
   - 3 (The user asked about the SmartX Pro phone, the Fotosnap camera, and TV or TV-related products.)

5. **For each question that the user asked, is there a corresponding answer to it?**
   - Question 1 (about the SmartX Pro phone): Y
   - Question 2 (about the Fotosnap camera): Y
   - Question 3 (about TVs or TV-related products): Y

6. **Of the number of questions asked, how many of these questions were addressed by the answer?** (output a number)
   - 3

The submitted answer accurately reflects the information provided in the context without adding any new information or contradicting 

Evaluate the LLM's answer to the user based on an "ideal" / "expert" (human generated) answer.

In [19]:
test_set_ideal = {
    'customer_msg': """\
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs or TV related products do you have?""",
    'ideal_answer':"""\
Of course!  The SmartX ProPhone is a powerful \
smartphone with advanced camera features. \
For instance, it has a 12MP dual camera. \
Other features include 5G wireless and 128GB storage. \
It also has a 6.1-inch display.  The price is $899.99.

The FotoSnap DSLR Camera is great for \
capturing stunning photos and videos. \
Some features include 1080p video, \
3-inch LCD, a 24.2MP sensor, \
and interchangeable lenses. \
The price is 599.99.

For TVs and TV related products, we offer 3 TVs \


All TVs offer HDR and Smart TV.

The CineView 4K TV has vibrant colors and smart features. \
Some of these features include a 55-inch display, \
'4K resolution. It's priced at 599.

The CineView 8K TV is a stunning 8K TV. \
Some features include a 65-inch display and \
8K resolution.  It's priced at 2999.99

The CineView OLED TV lets you experience vibrant colors. \
Some features include a 55-inch display and 4K resolution. \
It's priced at 1499.99.

We also offer 2 home theater products, both which include bluetooth.\
The SoundMax Home Theater is a powerful home theater system for \
an immmersive audio experience.
Its features include 5.1 channel, 1000W output, and wireless subwoofer.
It's priced at 399.99.

The SoundMax Soundbar is a sleek and powerful soundbar.
It's features include 2.1 channel, 300W output, and wireless subwoofer.
It's priced at 199.99

Are there any questions additional you may have about these products \
that you mentioned here?
Or may do you have other questions I can help you with?
    """
}

Check if the LLM's response agrees with or disagrees with the expert answer
This evaluation prompt is from the OpenAI evals project.

BLEU score: another way to evaluate whether two pieces of text are similar or not.

In [20]:
def eval_vs_ideal(test_set, assistant_answer):

    cust_msg = test_set['customer_msg']
    ideal = test_set['ideal_answer']
    completion = assistant_answer

    system_message = """\
    You are an assistant that evaluates how well the customer service agent \
    answers a user question by comparing the response to the ideal (expert) response
    Output a single letter and nothing else.
    """

    user_message = f"""\
You are comparing a submitted answer to an expert answer on a given question. Here is the data:
    [BEGIN DATA]
    ************
    [Question]: {cust_msg}
    ************
    [Expert]: {ideal}
    ************
    [Submission]: {completion}
    ************
    [END DATA]

Compare the factual content of the submitted answer with the expert answer. Ignore any differences in style, grammar, or punctuation.
    The submitted answer may either be a subset or superset of the expert answer, or it may conflict with it. Determine which case applies. Answer the question by selecting one of the following options:
    (A) The submitted answer is a subset of the expert answer and is fully consistent with it.
    (B) The submitted answer is a superset of the expert answer and is fully consistent with it.
    (C) The submitted answer contains all the same details as the expert answer.
    (D) There is a disagreement between the submitted answer and the expert answer.
    (E) The answers differ, but these differences don't matter from the perspective of factuality.
  choice_strings: ABCDE
"""

    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_message}
    ]

    response = get_completion_from_messages(messages)
    return response

In [21]:
print(assistant_answer)

The SmartX Pro Phone is a high-performance smartphone with a 6.1-inch display, 128GB storage, and a 12MP camera, priced at $999.

The Fotosnap Camera is a professional DSLR camera with a 24MP sensor, 4K video capability, and interchangeable lenses, priced at $1200.

As for TVs or TV-related products, we have a range of smart TVs with 4K resolution, HDR, and smart TV features, priced between $700 and $2000.


In [22]:
eval_vs_ideal(test_set_ideal, assistant_answer)

'D'

In [23]:
assistant_answer_2 = "life is like a box of chocolates"

In [24]:
eval_vs_ideal(test_set_ideal, assistant_answer_2)

'D'